In [7]:
import pandas as pd
import sqlite3

# Load data
portfolio = pd.read_csv("../data/processed/portfolio.csv")
portfolio["age_group"] = pd.cut(
    portfolio["age"],
    bins=[20, 30, 40, 50, 60, 70, 80],
    labels=["20-30", "30-40", "40-50", "50-60", "60-70", "70-80"])

portfolio["expected_claim"] = (
    portfolio["expected_death_probability"] * portfolio["sum_insured"])

# Create in-memory database
conn = sqlite3.connect(":memory:")

# Push data to SQL
portfolio.to_sql("portfolio", conn, index=False, if_exists="replace")

10000

In [8]:
query = """
SELECT 
    age_group,
    SUM(expected_death_probability) AS expected_deaths,
    SUM(actual_death) AS actual_deaths,
    SUM(actual_death) * 1.0 / SUM(expected_death_probability) AS ae_ratio
FROM portfolio
GROUP BY age_group
"""

pd.read_sql(query, conn)

,age_group,expected_deaths,actual_deaths,ae_ratio
0,None,0.07065,0,0.000000
1,20-30,0.97028,2,2.061261
2,30-40,1.49630,3,2.004946
3,40-50,2.79420,4,1.431537
4,50-60,6.28464,8,1.272945
5,60-70,15.92272,18,1.130460
6,70-80,44.94132,44,0.979054


In [9]:
query = """
SELECT 
    sex,
    SUM(expected_death_probability) AS expected_deaths,
    SUM(actual_death) AS actual_deaths,
    SUM(actual_death) * 1.0 / SUM(expected_death_probability) AS ae_ratio
FROM portfolio
GROUP BY sex
"""

pd.read_sql(query, conn)

,sex,expected_deaths,actual_deaths,ae_ratio
0,female,29.05934,29,0.997958
1,male,43.42077,50,1.151523


In [10]:
query = """
SELECT 
    age_group,
    SUM(sum_insured * expected_death_probability) AS expected_claims,
    SUM(sum_insured * actual_death) AS actual_claims
FROM portfolio
GROUP BY age_group
"""

pd.read_sql(query, conn)

,age_group,expected_claims,actual_claims
0,None,3.581261e+04,0
1,20-30,5.085137e+05,868454
2,30-40,7.908009e+05,2294086
3,40-50,1.456596e+06,2675460
4,50-60,3.321645e+06,4245591
5,60-70,8.398334e+06,10306538
6,70-80,2.278892e+07,23309669


## SQL-Based Analysis

This section demonstrates the use of SQL for data aggregation and experience analysis.

SQL queries were used to replicate actuarial calculations such as A/E ratios and claim summaries, highlighting the ability to perform data transformation and analysis across different tools.